# Démonstration : Interactions de Shapley avec AntakIA

Ce notebook démontre le calcul **rapide** des interactions d'ordre 2 via SHAP natif.

## Rappel Conceptuel

> **Aphorisme Directeur** : *"Là où il y a interaction forte, il y a structure causale à découvrir."*

Les interactions de Shapley révèlent où le modèle n'est **pas additif** :
- **SI(i,j) > 0** : Synergie (i et j ensemble font plus que la somme)
- **SI(i,j) < 0** : Redondance (i et j se substituent)
- **SI(i,j) ≈ 0** : Indépendance (effets additifs)

In [ ]:
# Configuration
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

PROJECT_ROOT = Path().absolute().parent
src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")

## 1. Données et Modèle

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import fetch_california_housing

# Charger California Housing
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Sous-échantillonnage
X_sample = X.iloc[:1000]
y_sample = y[:1000]

print(f"Dataset: {X_sample.shape[0]} observations, {X_sample.shape[1]} features")
print(f"Features: {list(X_sample.columns)}")

In [ ]:
# Entraîner RandomForest
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
model.fit(X_sample, y_sample)
print(f"R² train: {model.score(X_sample, y_sample):.3f}")

## 2. Calcul des Interactions (MÉTHODE RAPIDE)

Utilisation de `compute_shap_interactions_fast()` qui utilise SHAP TreeExplainer natif.

**~100x plus rapide** que shapiq pour les modèles arbres !

In [ ]:
from antakia.interactions import (
    compute_shap_interactions_fast,
    is_tree_model,
    get_interaction_matrix,
    identify_problematic_features,
)

# Vérifier compatibilité
print(f"Modèle compatible TreeExplainer: {is_tree_model(model)}")
print("→ Méthode rapide disponible !")

In [ ]:
%%time
# Calcul RAPIDE des interactions
result = compute_shap_interactions_fast(
    model,
    X_sample.values,
    n_samples=200,  # 200 samples en quelques secondes !
    use_gpu=False,  # True si GPU disponible
    verbose=True
)

## 3. Analyse des Résultats

In [ ]:
# Top 10 interactions
print("Top 10 Interactions (par moyenne |SI|):")
print("-" * 50)
for (i, j), val in result.get_top_interactions(10):
    feat_i = X_sample.columns[i]
    feat_j = X_sample.columns[j]
    print(f"  {feat_i:12} × {feat_j:12} : {val:.4f}")

In [ ]:
# Features problématiques (fortes interactions)
problematic_idx, scores = identify_problematic_features(result, threshold_percentile=80)

print(f"\nFeatures avec fortes interactions (top 20%):")
for idx in problematic_idx:
    print(f"  - {X_sample.columns[idx]}: score = {scores[idx]:.4f}")

print("\n💡 Ces features sont candidates pour :")
print("   - Découpage spatial (GRANITE)")
print("   - Formation de coalitions sémantiques")

## 4. Visualisation

In [ ]:
from antakia.interactions import (
    plot_interaction_heatmap,
    plot_feature_interaction_scores,
)

# Heatmap des interactions
fig = plot_interaction_heatmap(
    result,
    feature_names=list(X_sample.columns),
    title="Interactions de Shapley - California Housing"
)
fig.show()

In [ ]:
# Scores par feature
fig = plot_feature_interaction_scores(
    result,
    feature_names=list(X_sample.columns),
    highlight_problematic=True
)
fig.show()

## 5. Indices de Coopétition

Les indices de coopétition mesurent si une feature est plutôt **coopérative** ou **compétitive**.

In [ ]:
from antakia.coopetition import (
    compute_coopetition_indices,
    compute_coalition_candidates,
    plot_coopetition_spectrum,
    plot_cooperative_network,
)

# Calculer les indices de coopétition
coopetition = compute_coopetition_indices(
    result,
    feature_names=list(X_sample.columns),
    verbose=True
)

In [ ]:
# Spectre de coopétition
fig = plot_coopetition_spectrum(coopetition)
fig.show()

In [ ]:
# Réseau coopératif
fig = plot_cooperative_network(coopetition, min_edge_weight=0.005)
fig.show()

In [ ]:
# Propositions de coalitions
coalitions = compute_coalition_candidates(
    coopetition, result,
    min_cooperation=0.01,
    verbose=True
)

## 6. Interprétation Causale

### Rappel : Interactions → Causalité

| Pattern | Structure | Exemple |
|---------|-----------|--------|
| SI(Lat, Lon) > 0 | Cause commune | Position géographique |
| SI(MedInc, AveOccup) | Médiation | Revenu → Occupation → Prix |
| SI(AveRooms, HouseAge) | Modération | Âge modère effet des pièces |

In [ ]:
# Matrice d'interaction
matrix = get_interaction_matrix(result, aggregation='mean_abs')

# Interaction (Latitude × Longitude)
lat_idx = list(X_sample.columns).index('Latitude')
lon_idx = list(X_sample.columns).index('Longitude')

print(f"Interaction (Latitude × Longitude): {matrix[lat_idx, lon_idx]:.4f}")
print("\n💡 Cette interaction suggère une CAUSE COMMUNE : la localisation géographique.")
print("   → Candidat pour une COALITION SÉMANTIQUE : 'position_géographique'")

## Conclusion

Ce notebook a démontré :

1. **Calcul RAPIDE** via SHAP TreeExplainer (~100x plus rapide que shapiq)
2. **Visualisation** : heatmap, bar chart, réseau
3. **Indices de Coopétition** : coopératif vs compétitif
4. **Propositions de Coalitions** : variables latentes candidates
5. **Interprétation Causale** : interactions → structures causales

### Prochaines Étapes

- **Sprint 3** : Clustering Dyadique (ToMATo amélioré)
- **Sprint 4** : Moteur de Tessellation
- **GPU Scaleway** : Calculs massivement parallèles